In [1]:
!pip install matplotlib seaborn pandas streamlit --break-system-packages 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 1.2 MB/s  0:00:08m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.2/731.2 kB 1.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 1.1 MB/s  0:00:06 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 1.1 MB/s  0:00:41m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [streamlit]14 [streamlit]


In [11]:
%matplotlib inline
import os
import gc

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [51]:
def read_csv(file:str) -> pd.DataFrame:
    ENCODING="windows-1252"
    SEP=";"
    return pd.read_csv(file, encoding=ENCODING, sep=SEP)

In [4]:
csv_files = list(filter(lambda file: 'csv' in file, os.listdir()))
csv_files

['base_chegadas_2025_acum_08_agosto.csv',
 'chegadas_2023.csv',
 'chegadas_2024.csv']

In [52]:
data_2025 = read_csv(csv_files[0])
data_2025.head()

,Via_de_acesso,UF,nome_pais_correto,mes,ano,Chegadas
0,Aérea,Rio de Janeiro,Afeganistão,Janeiro,2025,1
1,Aérea,São Paulo,Afeganistão,Janeiro,2025,3
2,Terrestre,Paraná,Afeganistão,Janeiro,2025,2
3,Terrestre,Santa Catarina,Afeganistão,Janeiro,2025,4
4,Terrestre,Santa Catarina,Afeganistão,Janeiro,2025,2


In [53]:
data_2023 = read_csv(csv_files[1])
data_2023.head()

,Continente,cod continente,País,cod pais,UF,cod uf,Via,cod via,ano,Mês,cod mes,Chegadas
0,América do Sul,4,Peru,34,Acre,1,Aéreo,1,2023,Novembro,11,0
1,América do Sul,4,Peru,34,Acre,1,Aéreo,1,2023,Novembro,11,0
2,Ásia,5,Outros países,56,Outras Unidades da Federação,99,Aéreo,1,2023,Junho,6,0
3,Europa,6,Alemanha,57,Outras Unidades da Federação,99,Aéreo,1,2023,Janeiro,1,12
4,Europa,6,Alemanha,57,Outras Unidades da Federação,99,Aéreo,1,2023,Fevereiro,2,10


In [54]:
data_2024 = read_csv(csv_files[2])
data_2024.head()

,Continente,cod continente,País,cod pais,UF,cod uf,Via,cod via,ano,Mês,cod mes,Chegadas
0,Europa,6,Alemanha,57,Outras Unidades da Federação,99,Aérea,1,2024,Janeiro,1,10
1,América do Sul,4,Argentina,26,Outras Unidades da Federação,99,Aérea,1,2024,Janeiro,1,1197
2,Europa,6,Bélgica,59,Outras Unidades da Federação,99,Aérea,1,2024,Janeiro,1,3
3,América do Norte,3,Canadá,23,Outras Unidades da Federação,99,Aérea,1,2024,Janeiro,1,2
4,América do Sul,4,Chile,28,Outras Unidades da Federação,99,Aérea,1,2024,Janeiro,1,1


In [85]:
countries_2025 = pd.Series(data_2025.nome_pais_correto.unique()).sort_values()
countries_2023 = pd.Series(data_2023["País"].unique()).sort_values()
countries_2024 = pd.Series(data_2024["País"].unique()).sort_values()

all_countries = pd.concat([countries_2023, countries_2024, countries_2025], axis=0)
print("before=%d"%(all_countries.shape[0]))
all_countries.drop_duplicates(inplace=True)
print("after=%d"%(all_countries.shape[0]))
all_countries.head()

before=383
after=207


2           Alemanha
51            Angola
3          Argentina
52    Arábia Saudita
4          Austrália
dtype: object

In [90]:
countries_continents_2024 = data_2024[["Continente","País"]]
countries_continents_2023 = data_2023[["Continente","País"]]

matching_continets_per_country = pd.concat([countries_continents_2024,countries_continents_2023],axis=0)
print("Before=%d"%(matching_continets_per_country.shape[0]))
matching_continets_per_country.drop_duplicates(inplace=True)
print("After=%d"%(matching_continets_per_country.shape[0]))

Before=57620
After=98


In [68]:
paises = pd.DataFrame(columns=["2023", "2024", "2025"])
paises["2023"] = paises_2023
paises["2024"] = paises_2024
paises["2025"] = paises_2025
paises.head()

,2023,2024,2025
2,Alemanha,Bélgica,Alemanha
51,Angola,República Dominicana,Emirados Árabes Unidos
3,Argentina,Canadá,Andorra
52,Arábia Saudita,República Tcheca,Equador
4,Austrália,Chile,Angola


In [58]:
columns_to_drop = ["cod uf", "cod continente", "cod mes", "cod pais", "cod via"]
rename_columns = {"Via_de_acesso":"Via", "nome_pais_correto":"País"}

data = pd.DataFrame()

for csv_file in csv_files:
    file = pd.read_csv(csv_file, encoding="windows-1252", sep=";")

    try:
        file.drop(columns_to_drop,inplace=True,axis=1)
    except KeyError:
        pass

    file.rename(rename_columns,inplace=True,axis="columns")

    print("%d file has rows=%d; columns=%d"%(year, file.shape[0], file.shape[1]))
    print("columns=%s\n"%(str(file.columns)))
    
    data = pd.concat([data,file],axis=0)
    del file
    gc.collect()

print("all_data file has rows=%d; columns=%d"%(data.shape[0], data.shape[1])) 
print("columns=%s"%(str(data.columns)))
data.to_csv("all_data.csv",index=False)

2024 file has rows=16271; columns=6
columns=Index(['Via', 'UF', 'País', 'mes', 'ano', 'Chegadas'], dtype='object')

2024 file has rows=34764; columns=7
columns=Index(['Continente', 'País', 'UF', 'Via', 'ano', 'Mês', 'Chegadas'], dtype='object')

2024 file has rows=22856; columns=7
columns=Index(['Continente', 'País', 'UF', 'Via', 'ano', 'Mês', 'Chegadas'], dtype='object')

all_data file has rows=73891; columns=8
columns=Index(['Via', 'UF', 'País', 'mes', 'ano', 'Chegadas', 'Continente', 'Mês'], dtype='object')
